# ConceptGraphs head-to-head — Colab (L4 recommended)

Scores **ConceptGraphs and our VSA memory on the same Replica scenes,
under ConceptGraphs' own evaluation** (mAcc / F-mIoU, their scorer,
their ground truth). No invented metrics: our trace is made to emit the
same output type their eval consumes, so one scorer judges both.

**Runtime choice matters here:**

| runtime | VRAM | backbone it can run | ~units/h | verdict |
|---|---|---|---|---|
| **L4** | 24 GB | **SAM ViT-H — their default** | ~4.8 | **use this** |
| T4 | 16 GB | MobileSAM (supported variant) | ~1.8 | works, config differs |
| TPU | — | none of this stack | — | not usable (see below) |

*Why not the TPUs:* every model here (YOLO/ultralytics, SAM,
GroundingDINO, CLIP, EigenPlaces) is CUDA-path PyTorch. TPUs need
XLA-compiled graphs; porting an inference pipeline over a few thousand
images would cost far more effort than the compute it saves. Spend the
TPU allocation elsewhere.

**Budget:** one scene ≈ 30–90 min on L4 ≈ **3–7 units**. All 8 Replica
scenes ≈ 25–55 units. Run `room0` first (it is ConceptGraphs' own demo
scene) and only continue if its mAcc lands near their published ~0.40.


## 0 · Runtime check + unit estimate


In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi',
                      '--query-gpu=name,memory.total',
                      '--format=csv'], capture_output=True,
                     text=True).stdout)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
GSA = 'sam_vit_h' if vram >= 20 else ('mobilesam' if vram >= 6 else None)
print(f'VRAM {vram:.0f} GB -> segmentation backbone: {GSA}')
assert GSA, 'No usable GPU — Runtime > Change runtime type > L4 (or T4)'
if GSA != 'sam_vit_h':
    print('NOTE: not their default config — record this in the writeup.')


## 1 · Drive workspace

Repo, their repo, ~6 GB of checkpoints and the scene data all live in
Drive. A disconnect then costs minutes, not a re-download.


In [ ]:
from google.colab import drive; from pathlib import Path; import os
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/ssnslam_colab')
WORKSPACE = DRIVE / 'workspace'; HANDOFF = DRIVE / 'handoff'
for p in (WORKSPACE, HANDOFF): p.mkdir(parents=True, exist_ok=True)
print(WORKSPACE)


## 2 · Our repo + deps


In [ ]:
import subprocess, os
REPO_URL = 'https://github.com/SynapticScotsman/Semantic-Spiking-Neural-SLAM-2023.git'; BRANCH = 'results-sites'
# repo on the LOCAL VM disk: a git tree on Drive-FUSE corrupts (seen:
# fetch ok but 'reset --hard origin/BRANCH' exit 128 on a stale clone)
# and is slow. It is ~50 MB — a fresh shallow clone costs seconds, and
# it guarantees the exact branch tip every session.
import shutil
REPO_DIR = Path('/content/Semantic-Spiking-Neural-SLAM-2023')
if (REPO_DIR / '.git').exists():
    r = subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin',BRANCH])
    if r.returncode == 0:
        r = subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard','FETCH_HEAD'])
    if r.returncode != 0:
        print('existing clone unusable — re-cloning'); shutil.rmtree(REPO_DIR)
if not (REPO_DIR / '.git').exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--depth','1',
                    REPO_URL,str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
# outputs (small, valuable) live on Drive; RAW scene data lives on the
# LOCAL VM disk — Drive-backed writes make the 5 GB fetch and the YOLO
# read loop several times slower, and prepare_replica's resumable
# range-fetch re-pulls a lost scene in minutes on Colab bandwidth.
tgt = WORKSPACE / 'outputs'; tgt.mkdir(parents=True, exist_ok=True)
# the repo TRACKS outputs/ (committed html/json artifacts), so a fresh
# clone ALWAYS has a real directory here and a bare exists() check
# silently skipped the symlink — stage-4 intermediates (gt_instances,
# detections_crops, object_points) then sat on the ephemeral VM disk
# and were lost on disconnect, which is GPU-minutes, not seconds.
# Merge the tracked files into Drive (-n never clobbers a newer copy
# there), then replace the directory with the link.
if not os.path.islink('outputs'):
    if os.path.isdir('outputs'):
        subprocess.run(['cp','-rn','outputs/.',str(tgt)], check=True)
        shutil.rmtree('outputs')
    os.symlink(tgt, 'outputs')
local = Path('/content/replica_data'); local.mkdir(parents=True, exist_ok=True)
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/replica'): os.symlink(local, 'data/replica')
!pip -q install ultralytics transformers scipy 2>&1 | tail -1
# parallel downloader — the scene fetch uses it when present (8 connections beat one wget stream)
!apt-get -qq install -y aria2 > /dev/null 2>&1 || true
print('repo ready at', os.getcwd())
print('outputs ->', os.path.realpath('outputs'),
      '(symlink:', os.path.islink('outputs'), ')')
print('data/replica ->', os.path.realpath('data/replica'), '(local, ephemeral)')


## 3 · ConceptGraphs + checkpoints (class-agnostic variant)

We run **class-agnostic ConceptGraphs** (`class_set none`) — a headline
row of their paper that needs only SAM, skipping the GroundingDINO CUDA
compile and the 5.6 GB RAM tagger. Every install is LOUD: a failed step
raises here instead of surfacing three cells later.

One-time per VM: chamferdist + gradslam compile in minutes; **pytorch3d
compiles from source (~15–25 min)** unless a matching wheel exists.
Everything is skip-if-present on re-runs of the same VM.


In [ ]:
import os, subprocess, time
from pathlib import Path

def pip(*args, label=None):
    t0 = time.time()
    r = subprocess.run(['pip','install','-q',*args])
    print(f"pip {label or ' '.join(args)[:60]}: exit {r.returncode} in {time.time()-t0:.0f}s", flush=True)
    if r.returncode != 0:
        raise RuntimeError(f'pip install failed: {args} — scroll up for the real error')

CG = WORKSPACE / 'concept-graphs'
GSA_DIR = WORKSPACE / 'Grounded-Segment-Anything'
if not (CG / '.git').exists():
    subprocess.run(['git','clone',
        'https://github.com/concept-graphs/concept-graphs.git',str(CG)], check=True)
if not (GSA_DIR / '.git').exists():
    subprocess.run(['git','clone',
        'https://github.com/IDEA-Research/Grounded-Segment-Anything.git',str(GSA_DIR)], check=True)
os.environ['GSA_PATH'] = str(GSA_DIR)   # their scripts require this env var

# supervision pinned: newer versions renamed ColorPalette.default()
# and the annotator API their utils/vis.py depends on. 0.17.1 is the
# oldest release that installs on py3.12 AND still carries their API
# (all five of their vis.py usages verified against it).
pip('tyro','open_clip_torch','h5py','hydra-core','distinctipy','ultralytics',
    'supervision==0.17.1','open3d','faiss-cpu','openai','imageio', label='core deps')
# supervision 0.17.1 caps opencv-python-headless at a 2023 build that
# was compiled against numpy 1 — it replaces Colab's numpy-2-built cv2
# and every cv2 import dies with '_ARRAY_API not found'. Restore a
# modern build; supervision runs fine on it (the cap was conservative).
pip('--upgrade','--no-deps','opencv-python-headless',
    label='cv2 restore (undo supervision downgrade)')
pip('segment-anything', label='segment-anything')
pip('git+https://github.com/krrish94/chamferdist.git', label='chamferdist (compiles, ~2 min)')
pip('git+https://github.com/gradslam/gradslam.git@conceptfusion', label='gradslam@conceptfusion')
# pytorch3d: 25-30 min source compile. Cache the built package to
# Drive so ONLY THE FIRST VM ever pays it; later VMs restore in
# seconds. Cache is keyed by torch version — a Colab torch upgrade
# triggers one honest recompile.
import torch as _torch
_SP = '/usr/local/lib/python3.12/dist-packages'
P3D_TAR = DRIVE / 'wheels' / f"pytorch3d_py312_torch{_torch.__version__.split('+')[0]}.tgz"
try:
    import pytorch3d
    print('pytorch3d already importable')
except Exception:
    if P3D_TAR.exists():
        print('restoring pytorch3d from Drive cache (seconds, not 30 min)')
        subprocess.run(['tar','xzf',str(P3D_TAR),'-C',_SP], check=True)
        import importlib; importlib.invalidate_caches()
    else:
        pip('git+https://github.com/facebookresearch/pytorch3d.git',
            label='pytorch3d (SOURCE COMPILE ~25-30 min — cached to Drive after)')
        P3D_TAR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['bash','-c',
            f"cd {_SP} && tar czf '{P3D_TAR}' pytorch3d pytorch3d-*.dist-info"],
            check=True)
        print(f'pytorch3d cached to Drive ({P3D_TAR.stat().st_size/1e6:.0f} MB) for future VMs')
pip('-e', str(CG), label='conceptgraphs editable')

# import their vis module too — it is where library-API drift
# (supervision et al.) surfaces; better here than mid-pipeline
r = subprocess.run(['python','-c',
    'import conceptgraph, gradslam, chamferdist, conceptgraph.utils.vis'],
                   capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError('post-install import check failed:' + chr(10) + r.stderr[-2000:])
print('conceptgraph + gradslam + chamferdist + utils.vis import OK')

# checkpoints go where THEIR loader looks: GSA_PATH root / EfficientSAM/
def get(url, rel):
    p = GSA_DIR / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    if not p.exists():
        print('downloading', rel); subprocess.run(['wget','-q','-O',str(p),url], check=True)
    print(f'{rel}: {p.stat().st_size/1e6:.0f} MB')
get('https://github.com/ChaoningZhang/MobileSAM/raw/master/weights/mobile_sam.pt',
    'EfficientSAM/mobile_sam.pt')
if GSA == 'sam_vit_h':
    get('https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth',
        'sam_vit_h_4b8939.pth')


## 4 · Scene data + our observations

Same commands as the local pipeline — RGB-D + poses, GT renders, then
our YOLO detections and world placement (all GPU-accelerated here).


In [ ]:
SCENE = 'room0'   # ConceptGraphs' own demo scene — start here

import os, subprocess, time, datetime, threading
# A runtime restart resets the working directory but not the open
# notebook — the single most common way this cell fails with exit 2
# ('can't open file') / exit 1 ('no module named vsa_cognitive_mapping').
assert os.path.exists('vsa_cognitive_mapping'), (
    'Kernel is not inside the repo (runtime was restarted?). '
    'Run cells 1-3 first — cell 2 clones/updates the repo and chdirs into it.')
assert 'WORKSPACE' in globals(), 'run cell 1 first (Drive mount defines WORKSPACE)'

# Every stage streams its output here AND mirrors it to a log on Drive,
# so progress is checkable from drive.google.com on any device without
# touching this page. The … heartbeat proves a quiet stage is still alive.
LOG = f'{WORKSPACE}/run_log_{SCENE}.txt'
def _log(line):
    with open(LOG, 'a') as f: f.write(line + chr(10))
def sh(c, cwd=None):
    cmd = ' '.join(map(str, c))
    stamp = datetime.datetime.now().strftime('%H:%M:%S')
    print(f'[{stamp}] $ {cmd}', flush=True); _log(f'[{stamp}] $ {cmd}')
    t0 = time.time(); last = [t0]
    env = dict(os.environ, PYTHONUNBUFFERED='1')
    p = subprocess.Popen(c, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, errors='replace', env=env, cwd=cwd)
    def pulse():
        while p.poll() is None:
            time.sleep(30)
            if p.poll() is None and time.time() - last[0] > 300:
                m = int((time.time() - t0) / 60)
                msg = f'  … still running ({m} min elapsed, no new output — normal for long model passes)'
                print(msg, flush=True); _log(msg); last[0] = time.time()
    threading.Thread(target=pulse, daemon=True).start()
    with open(LOG, 'a') as f:
        for ln in p.stdout:
            ln = ln.rstrip()
            print(ln, flush=True); f.write(ln + chr(10)); f.flush(); last[0] = time.time()
    rc = p.wait()
    done = f'  exit {rc} in {time.time()-t0:.0f}s'
    print(done, flush=True); _log(done)
    if rc != 0:
        raise RuntimeError(f'stage failed: {cmd} — full output in {LOG}; every later cell depends on this stage')
    return True

cfg = f'vsa_cognitive_mapping/configs/replica_{SCENE}.json'
from pathlib import Path
if not Path(f'data/replica/{SCENE}/poses.csv').exists():
    sh(['python','tools/prepare_replica.py','--scene',SCENE])
if not Path(f'outputs/replica_{SCENE}/gt_instances.json').exists():
    sh(['python','tools/replica_gt_from_renders.py','--scene',SCENE])
if not Path(f'outputs/replica_{SCENE}/detections_crops.csv').exists():
    sh(['python','-m','vsa_cognitive_mapping.classroom_pipeline',
        'embed-crops','--dataset',cfg])
if not Path(f'outputs/replica_{SCENE}/object_points.json').exists():
    sh(['python','-m','vsa_cognitive_mapping.object_grounding','--dataset',cfg,
        '--gt-json',f'outputs/replica_{SCENE}/gt_instances.json'])
sh(['python','student_gpu_package/01_check_data.py','--scene',SCENE])


## 5 · SMOKE their pipeline first (~40 frames)

Spend two minutes proving the install before spending units on 2000
frames. If this fails, the fix is in their README — nothing below is
worth attempting until it passes.


In [ ]:
import os, subprocess, glob
assert os.path.exists('vsa_cognitive_mapping'), 'run cells 1-4 first (runtime restart lost the working directory)'
# 01_check_data builds the layout on local VM disk on Colab (Drive
# FUSE makes 4000-file layouts and per-frame reads painfully slow)
CG_DATA = '/content/cg_dataset' if os.path.isdir('/content') else 'student_gpu_package/cg_dataset'

# Precondition: cell 4 must have completed (it ends by building the
# cg_dataset layout). Self-heal once; explain precisely if that fails.
if not os.path.exists(f'{CG_DATA}/{SCENE}/traj.txt'):
    print('cg_dataset missing — running the data check to (re)build it...')
    r = subprocess.run(['python','student_gpu_package/01_check_data.py',
                        '--scene',SCENE])
    if r.returncode != 0 or not os.path.exists(f'{CG_DATA}/{SCENE}/traj.txt'):
        raise RuntimeError(
            'cell 4 did not complete for this scene. Re-run cell 4 and '
            'watch its per-stage exit codes — with a fresh VM the local '
            'scene data is gone and the fetch stage must run again '
            '(resumable, minutes on Colab bandwidth).')

SMOKE = f'{CG_DATA}_smoke/{SCENE}/results'
os.makedirs(SMOKE, exist_ok=True)
src = sorted(glob.glob(f'{CG_DATA}/{SCENE}/results/frame*.jpg'))[:40]
for p in src:
    d = os.path.join(SMOKE, os.path.basename(p))
    if not os.path.exists(d): os.symlink(os.path.abspath(p), d)
    dp = p.replace('frame','depth').replace('.jpg','.png')
    dd = os.path.join(SMOKE, os.path.basename(dp))
    if os.path.exists(dp) and not os.path.exists(dd): os.symlink(os.path.abspath(dp), dd)
import shutil
shutil.copy(f'{CG_DATA}/{SCENE}/traj.txt', f'{CG_DATA}_smoke/{SCENE}/traj.txt')
shutil.copy(f'{CG_DATA}/{SCENE}/cam_params.json', f'{CG_DATA}_smoke/{SCENE}/cam_params.json')
print(f'{len(src)} frames staged for smoke at {SMOKE}')

# their documented two-step flow, run from inside their repo:
#   1. generate_gsa_results.py  (SAM segmentation, --flag args)
#   2. cfslam_pipeline_batch.py (3D mapping, hydra key=value args)
import time
CONF = str(CG / 'conceptgraph/dataset/dataconfigs/replica/replica.yaml')
CGDIR = str(CG / 'conceptgraph')
SAM_VARIANT = 'sam' if GSA == 'sam_vit_h' else 'mobilesam'
SMOKE_ROOT = os.path.abspath(f'{CG_DATA}_smoke')
t0 = time.time()
sh(['python','scripts/generate_gsa_results.py','--dataset_root',SMOKE_ROOT,
    '--dataset_config',CONF,'--scene_id',SCENE,'--class_set','none',
    '--sam_variant',SAM_VARIANT,'--stride','5'], cwd=CGDIR)
sh(['python','slam/cfslam_pipeline_batch.py',f'dataset_root={SMOKE_ROOT}',
    f'dataset_config={CONF}','stride=5',f'scene_id={SCENE}',
    'spatial_sim_type=overlap','mask_conf_threshold=0.95','match_method=sim_sum',
    'sim_threshold=1.2','dbscan_eps=0.1','gsa_variant=none','class_agnostic=True',
    'skip_bg=True','max_bbox_area_ratio=0.5','merge_interval=20',
    'merge_visual_sim_thresh=0.8','merge_text_sim_thresh=0.8',
    'save_suffix=smoke'], cwd=CGDIR)
mins = (time.time() - t0) / 60
print(f'smoke: {mins:.1f} min for 8 frames -> full run (400 frames) ~ {mins*50:.0f} min')
print(sorted(glob.glob(f'{SMOKE_ROOT}/{SCENE}/pcd_saves/*')))


## 6 · Full run on the scene (their verbatim README parameters)

Record the wall-clock — it is a column in the systems comparison.
Outputs land in the ephemeral dataset root, so the cell ends by copying
the pkl.gz results to Drive AND to where stage 3 reads them.


In [ ]:
import time, subprocess, os, glob, shutil
assert os.path.exists('vsa_cognitive_mapping'), 'run cells 1-5 first (runtime restart lost the working directory)'
ROOT_ABS = os.path.abspath(CG_DATA)
t0 = time.time()
sh(['python','scripts/generate_gsa_results.py','--dataset_root',ROOT_ABS,
    '--dataset_config',CONF,'--scene_id',SCENE,'--class_set','none',
    '--sam_variant',SAM_VARIANT,'--stride','5'], cwd=CGDIR)
sh(['python','slam/cfslam_pipeline_batch.py',f'dataset_root={ROOT_ABS}',
    f'dataset_config={CONF}','stride=5',f'scene_id={SCENE}',
    'spatial_sim_type=overlap','mask_conf_threshold=0.95','match_method=sim_sum',
    'sim_threshold=1.2','dbscan_eps=0.1','gsa_variant=none','class_agnostic=True',
    'skip_bg=True','max_bbox_area_ratio=0.5','merge_interval=20',
    'merge_visual_sim_thresh=0.8','merge_text_sim_thresh=0.8',
    'save_suffix=overlap_maskconf0.95_simsum1.2_dbscan.1_merge20_masksub'],
   cwd=CGDIR)
print(f'wall-clock {(time.time()-t0)/60:.1f} min | sam_variant {SAM_VARIANT}')

# persist: local disk dies with the VM; Drive + stage-3 location survive
SRC = f'{ROOT_ABS}/{SCENE}/pcd_saves'
OUT = f'{WORKSPACE}/cg_out/{SCENE}'
os.makedirs(OUT, exist_ok=True)
os.makedirs(f'student_gpu_package/cg_out/{SCENE}', exist_ok=True)
found = glob.glob(f'{SRC}/*.pkl.gz')
if not found:
    raise RuntimeError(f'no pkl.gz under {SRC} — the mapping step did not save; scroll up')
for p in found:
    shutil.copy(p, OUT); shutil.copy(p, f'student_gpu_package/cg_out/{SCENE}/')
    print('saved', os.path.basename(p), f'{os.path.getsize(p)/1e6:.1f} MB (Drive + cg_out)')


## 7 · Export, our labels, one scorer for both


In [ ]:
sh(['python','student_gpu_package/03_export_cg.py','--scene',SCENE])
sh(['python','student_gpu_package/04_vsa_labels.py','--scene',SCENE])
sh(['python','student_gpu_package/05_score.py','--scene',SCENE])
import json
print(json.dumps(json.load(open(
    f'student_gpu_package/handoff/{SCENE}/scores.json')), indent=2))


## 8 · Save the handoff

Sanity anchor before trusting anything: their room0 mAcc should land
near the paper's Replica average (~0.40). If it is wildly off, the
config drifted — send the run log rather than tuning toward the number.


In [ ]:
import shutil, time
stamp = time.strftime('%Y%m%d_%H%M')
dst = HANDOFF / f'{SCENE}_{stamp}'
shutil.copytree(f'student_gpu_package/handoff/{SCENE}', dst, dirs_exist_ok=True)
!pip list 2>/dev/null | grep -Ei 'torch|clip|segment|groundingdino|ultralytics' > {dst}/environment.txt
print('saved to', dst)
!du -sh {dst}


## 9 · What this costs and what to run next

- `room0` first (their demo scene, best chance of a clean install).
- If its mAcc is near ~0.40, queue the rest: `room1 room2 office0
  office1 office2 office3 office4` — roughly 25–55 units total on L4.
- Bring the `handoff/` folders home; scoring and every downstream
  analysis then runs locally in seconds.

**Pre-registered expectation (state it before seeing the number):** our
all-classes mAcc will be LOW — walls, floors and ceilings dominate the
point count and an object-detector-fed memory cannot label them, while
their SAM pipeline segments everything. Record which classes their eval
actually scores; the honest comparison is whatever their code reports,
not a subset we choose afterwards.
